# 99. Git first push

Revisa el estado del repositorio, permite definir un `.gitignore` mínimo, crear el primer commit y hacer el primer push a GitHub de forma controlada.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from pathlib import Path
import os
import subprocess
import shlex
import textwrap
import json

REPO_PATH = Path('/content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo')
assert REPO_PATH.exists(), f'No existe: {REPO_PATH}'

os.chdir(REPO_PATH)
print('Working dir:', Path.cwd())

Working dir: /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo


## Helpers

In [11]:
def run(cmd, check=True):
    print('$', cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Comando falló ({result.returncode}): {cmd}')
    return result

def git(cmd, check=True):
    return run(f'git {cmd}', check=check)

def file_exists(p):
    from pathlib import Path
    return Path(p).exists()

## 1. Revisión inicial del repo

In [12]:
git('rev-parse --is-inside-work-tree')
git('branch --show-current', check=False)
git('remote -v', check=False)
git('status --short')
git('status')
run('pwd')

$ git rev-parse --is-inside-work-tree
true

$ git branch --show-current
main

$ git remote -v
origin	https://github.com/eguivarjosed48-dot/llojeta-glacier-analysis.git (fetch)
origin	https://github.com/eguivarjosed48-dot/llojeta-glacier-analysis.git (push)

$ git status --short
 M .gitignore
 D notebooks/02_raster_intake
 D notebooks/03_raster_pairing_and_selection
 D notebooks/04_coregistration_nuth_kaab
 D notebooks/05_export_web_raster
 D notebooks/06_coregistration_evaluation
 D notebooks/07_coregistration_parameter_sweep
 D notebooks/99_git.ipynb
?? notebooks/02_raster_intake.ipynb
?? notebooks/03_raster_pairing_and_selection.ipynb
?? notebooks/04_coregistration_nuth_kaab.ipynb
?? notebooks/05_export_web_raster.ipynb
?? notebooks/06_coregistration_evaluation.ipynb
?? notebooks/07_coregistration_parameter_sweep.ipynb
?? notebooks/11_cleanup_and_finalize_selection.ipynb
?? notebooks/99_git_first_push.ipynb

$ git status
On branch main
Changes not staged for commit:
  (use "git add/r

CompletedProcess(args='pwd', returncode=0, stdout='/content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo\n', stderr='')

## 2. Revisar archivos pesados antes del primer push

In [13]:
run("find . -type f -size +25M | sort", check=False)

$ find . -type f -size +25M | sort
./data/raw/llojeta/2017/dsm/2017_DSM.tif
./data/raw/llojeta/2017/ortho/2017_Ortho.tif
./data/raw/llojeta/2018/dsm/2018_DSM.tif
./data/raw/llojeta/2018/ortho/2018_Ortho.tif
./data/raw/llojeta/2020/dsm/2020_DSM.tif
./data/raw/llojeta/2020/ortho/2020_Ortho.tif
./data/raw/llojeta/2025_B/dsm/2025_B_DSM.tif
./data/raw/llojeta/2025_B/ortho/2025_B_Ortho.tif
./outputs/coregistration_nk/2017_DSM_nk_corrected.tif
./outputs/coregistration_nk/2018_DSM_nk_corrected.tif
./outputs/coregistration_selected/2017_DSM_selected.tif
./outputs/coregistration_selected/2025_B_DSM_selected.tif
./outputs/coregistration_sweep/2017_vs_2025_B_factor_0p0.tif
./outputs/coregistration_sweep/2017_vs_2025_B_factor_m0p05.tif
./outputs/coregistration_sweep/2017_vs_2025_B_factor_m0p1.tif
./outputs/coregistration_sweep/2017_vs_2025_B_factor_m0p2.tif
./outputs/coregistration_sweep/2017_vs_2025_B_factor_m0p3.tif



CompletedProcess(args='find . -type f -size +25M | sort', returncode=0, stdout='./data/raw/llojeta/2017/dsm/2017_DSM.tif\n./data/raw/llojeta/2017/ortho/2017_Ortho.tif\n./data/raw/llojeta/2018/dsm/2018_DSM.tif\n./data/raw/llojeta/2018/ortho/2018_Ortho.tif\n./data/raw/llojeta/2020/dsm/2020_DSM.tif\n./data/raw/llojeta/2020/ortho/2020_Ortho.tif\n./data/raw/llojeta/2025_B/dsm/2025_B_DSM.tif\n./data/raw/llojeta/2025_B/ortho/2025_B_Ortho.tif\n./outputs/coregistration_nk/2017_DSM_nk_corrected.tif\n./outputs/coregistration_nk/2018_DSM_nk_corrected.tif\n./outputs/coregistration_selected/2017_DSM_selected.tif\n./outputs/coregistration_selected/2025_B_DSM_selected.tif\n./outputs/coregistration_sweep/2017_vs_2025_B_factor_0p0.tif\n./outputs/coregistration_sweep/2017_vs_2025_B_factor_m0p05.tif\n./outputs/coregistration_sweep/2017_vs_2025_B_factor_m0p1.tif\n./outputs/coregistration_sweep/2017_vs_2025_B_factor_m0p2.tif\n./outputs/coregistration_sweep/2017_vs_2025_B_factor_m0p3.tif\n', stderr='')

## 3. .gitignore mínimo recomendado

Este bloque está pensado para tu estado actual: evitar outputs voluminosos y temporales, pero conservar manifiestos, CSV/JSON finales y tablas HTML pequeñas.

In [14]:
gitignore_lines = [
    "# Python",
    "__pycache__/",
    "*.pyc",
    ".ipynb_checkpoints/",
    "",
    "# Large scientific outputs",
    "outputs/coregistration_nk/*.tif",
    "outputs/coregistration_sweep/",
    "outputs/coregistration_diagnostics/maps/",
    "outputs/coregistration_diagnostics/figures/",
    "",
    "# Optional bulky data layers",
    "data/interim/",
    "data/processed/",
    "data/publish/",
    "",
    "# Keep small reproducible outputs",
    "!outputs/coregistration_final/",
    "!outputs/coregistration_final/*.csv",
    "!outputs/coregistration_final/*.json",
    "!outputs/coregistration_diagnostics/*.csv",
    "!outputs/coregistration_diagnostics/*.json",
    "!outputs/qa/*.csv",
    "!outputs/qa/*.json",
    "!web/tables/",
    "!web/tables/*.html",
]

proposed_gitignore = "\n".join(gitignore_lines) + "\n"
print(proposed_gitignore)

# Python
__pycache__/
*.pyc
.ipynb_checkpoints/

# Large scientific outputs
outputs/coregistration_nk/*.tif
outputs/coregistration_sweep/
outputs/coregistration_diagnostics/maps/
outputs/coregistration_diagnostics/figures/

# Optional bulky data layers
data/interim/
data/processed/
data/publish/

# Keep small reproducible outputs
!outputs/coregistration_final/
!outputs/coregistration_final/*.csv
!outputs/coregistration_final/*.json
!outputs/coregistration_diagnostics/*.csv
!outputs/coregistration_diagnostics/*.json
!outputs/qa/*.csv
!outputs/qa/*.json
!web/tables/
!web/tables/*.html



## 4. Escribir o fusionar `.gitignore`

Pon `write_gitignore=True` para aplicarlo. Si ya tienes uno, lo fusiona conservando el contenido existente.

In [15]:
write_gitignore = True

gitignore_path = REPO_PATH / '.gitignore'

if write_gitignore:
    existing = gitignore_path.read_text(encoding='utf-8') if gitignore_path.exists() else ''
    existing_lines = existing.splitlines()
    merged = existing_lines[:]

    for line in proposed_gitignore.splitlines():
        if line not in merged:
            merged.append(line)

    gitignore_path.write_text("\n".join(merged).rstrip() + "\n", encoding='utf-8')
    print('Actualizado:', gitignore_path)
    print(gitignore_path.read_text(encoding='utf-8'))
else:
    print('write_gitignore=False → no se modifica .gitignore')

Actualizado: /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/.gitignore
# Python
__pycache__/
*.py[cod]
*.pyo
*.pyd

# Jupyter
.ipynb_checkpoints/

# OS
.DS_Store
Thumbs.db

# Data pesada
data/raw/
data/interim/
data/processed/

# Outputs temporales
outputs/

# Entornos
.env
.venv/
env/
venv/

# GIS auxiliares
*.aux.xml
*.ovr
*.tfw
*.prj~
*.pyc
# Large scientific outputs
outputs/coregistration_nk/*.tif
outputs/coregistration_sweep/
outputs/coregistration_diagnostics/maps/
outputs/coregistration_diagnostics/figures/
# Optional bulky data layers
data/publish/
# Keep small reproducible outputs
!outputs/coregistration_final/
!outputs/coregistration_final/*.csv
!outputs/coregistration_final/*.json
!outputs/coregistration_diagnostics/*.csv
!outputs/coregistration_diagnostics/*.json
!outputs/qa/*.csv
!outputs/qa/*.json
!web/tables/
!web/tables/*.html



## 5. Inicializar remote si aún no existe

Antes de ejecutar esta celda, crea un repo vacío en GitHub. Luego pega la URL HTTPS o SSH.

In [16]:
remote_name = 'origin'
remote_url = 'https://github.com/eguivarjosed48-dot/llojeta-glacier-analysis.git'  # ejemplo HTTPS: https://github.com/eguivarjosed48-dot/llojeta-glacier-analysis.git
                 # ejemplo SSH: git@github.com:eguivarjosed48-dot/llojeta-glacier-analysis.git

current_remote = git('remote -v', check=False)

if remote_url.strip():
    remotes = subprocess.run('git remote', shell=True, text=True, capture_output=True).stdout.split()
    if remote_name in remotes:
        git(f'remote set-url {shlex.quote(remote_name)} {shlex.quote(remote_url)}')
    else:
        git(f'remote add {shlex.quote(remote_name)} {shlex.quote(remote_url)}')
else:
    print('remote_url vacío → no se modifica remote')

$ git remote -v
origin	https://github.com/eguivarjosed48-dot/llojeta-glacier-analysis.git (fetch)
origin	https://github.com/eguivarjosed48-dot/llojeta-glacier-analysis.git (push)

$ git remote set-url origin https://github.com/eguivarjosed48-dot/llojeta-glacier-analysis.git


## 6. Revisión después de `.gitignore`

In [ ]:
git('status --short')
git('diff --stat', check=False)

$ git status --short
 M .gitignore
 D notebooks/02_raster_intake
 D notebooks/03_raster_pairing_and_selection
 D notebooks/04_coregistration_nuth_kaab
 D notebooks/05_export_web_raster
 D notebooks/06_coregistration_evaluation
 D notebooks/07_coregistration_parameter_sweep
 D notebooks/99_git.ipynb
?? notebooks/02_raster_intake.ipynb
?? notebooks/03_raster_pairing_and_selection.ipynb
?? notebooks/04_coregistration_nuth_kaab.ipynb
?? notebooks/05_export_web_raster.ipynb
?? notebooks/06_coregistration_evaluation.ipynb
?? notebooks/07_coregistration_parameter_sweep.ipynb
?? notebooks/11_cleanup_and_finalize_selection.ipynb
?? notebooks/99_git_first_push.ipynb

$ git diff --stat


## 7. Selección controlada de archivos para el primer commit

Deja vacío para usar `git add .` con el `.gitignore` ya aplicado. O especifica rutas manuales.

In [ ]:
files_to_add = """"""  # ejemplo:
# notebooks/09_coregistration_diagnostics.ipynb
# notebooks/10_coregistration_html_summary.ipynb
# notebooks/11_cleanup_and_finalize_selection.ipynb
# notebooks/99_git_first_push.ipynb
# src/
# config/
# outputs/coregistration_final/
# outputs/coregistration_diagnostics/coreg_diag_decision_table.csv
# outputs/coregistration_diagnostics/coreg_diag_epoch_summary.csv
# outputs/coregistration_diagnostics/coreg_diag_extended_stats.csv
# outputs/coregistration_diagnostics/coreg_diag_scenario_comparison.csv
# outputs/coregistration_diagnostics/coreg_diag_summary.json
# outputs/qa/project_raster_inventory.csv
# outputs/qa/project_raster_inventory.json
# web/tables/

git('reset', check=False)

if files_to_add.strip():
    paths = [line.strip() for line in files_to_add.splitlines() if line.strip() and not line.strip().startswith('#')]
    for p in paths:
        git(f'add {shlex.quote(p)}')
else:
    git('add .')

git('status --short')
git('diff --cached --stat', check=False)

## 8. Commit del primer snapshot

In [ ]:
commit_message = 'initial reproducible glacier coregistration pipeline snapshot'

diff_cached = git('diff --cached --quiet', check=False)
if diff_cached.returncode == 0:
    print('No hay cambios staged. No se crea commit.')
else:
    git(f'commit -m {shlex.quote(commit_message)}')
    git('log --oneline -n 3', check=False)

## 9. Primer push a GitHub

Para el primer push, normalmente necesitas:
- HTTPS + token personal, o
- SSH configurado en Colab/Drive

Configura `do_push=True` solo cuando el remote ya esté definido.

In [ ]:
do_push = False
remote_name = 'origin'

branch_name = subprocess.run(
    'git branch --show-current',
    shell=True,
    text=True,
    capture_output=True
).stdout.strip()

print('Remote:', remote_name)
print('Branch:', branch_name)

if do_push:
    git(f'push -u {shlex.quote(remote_name)} {shlex.quote(branch_name)}')
else:
    print('Push omitido. Cambia do_push=True para ejecutar el primer push.')

## 10. Snapshot final

In [ ]:
git('remote -v', check=False)
git('status')
git('log --oneline -n 5', check=False)

## Orden recomendado de uso

1. Ejecuta revisión inicial
2. Revisa archivos pesados
3. Aplica `.gitignore`
4. Configura `remote_url`
5. Revisa `status`
6. Haz `add` y `commit`
7. Cambia `do_push=True`
8. Ejecuta el primer push